In [1]:
import kaggle as kg
import os
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from matplotlib.pyplot import figure
from glob import glob
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img,img_to_array
from keras.layers import Input, Dense,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization
from keras.models import Model
from keras.optimizers import RMSprop,SGD
plt.style.use("ggplot")

2024-08-25 19:37:56.059333: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-25 19:37:56.128383: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-25 19:37:56.685398: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-25 19:37:56.689201: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-25 19:37:58.298114: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [2]:
# os.environ["KAGGLE_USERNAME"] = "thasinkhan"
# os.environ["KAGGLE_KEY"] = "d0e58b5e1972f61220788d36914159c2"

In [3]:
# kg.api.authenticate()

In [4]:
# kg.api.dataset_download_files(dataset="vipoooool/new-plant-diseases-dataset",
#                               path="dataset",unzip=True)

In [4]:
def count_folders(directory):
    folder_count = 0
    for entry in os.scandir(directory):
        if entry.is_dir():
            folder_count += 1
    return folder_count

# Function to count the total number of files in a directory
def total_files(folder_path):
    num_files = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
    return num_files


train = "/home/thasin/plant_disease/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/train"
valid = "/home/thasin/plant_disease/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/valid"
test = "/home/thasin/plant_disease/dataset/test/test"

# Count the number of folders in train and valid directories
total_folders_train = count_folders(train)
total_folders_valid = count_folders(valid)

# Count the total number of files in the test directory
total_files_test = total_files(test)

print(f"Total number of folders in Train: {total_folders_train}")
print(f"Total number of folders in Valid: {total_folders_valid}")
print(f"Total number of files in Test: {total_files_test}")


Total number of folders in Train: 38
Total number of folders in Valid: 38
Total number of files in Test: 33


In [5]:
import pathlib
import pandas as pd

def generate_image_dataframe(path, is_test=False):
    img_path = []
    img_label = []

    if is_test:
        # Iterate through each image in the test folder
        for img_file_path in pathlib.Path(path).glob("*.JPG"):
            img_path.append(str(img_file_path))  # Store the image path
            
            # Assuming label is derived from the filename, for example, before the first underscore
            img_label.append(str(img_file_path.stem).split("_")[0])
    else:
        # Iterate through each class directory
        for single_class_dir_path in pathlib.Path(path).glob("*"):
            # Iterate through each image in the class directory
            for single_class_img_path in pathlib.Path(single_class_dir_path).glob("*.jpg"):
                img_path.append(str(single_class_img_path))  # Store the image path
                # Assuming the label is the last part of the folder name split by an underscore
                img_label.append(str(single_class_img_path).split("/")[-2].split("_")[-1])

    # Return a DataFrame containing the image paths and corresponding labels
    return pd.DataFrame(data={"img_path": img_path, "label": img_label})

# Example usage:

train_path = "/home/thasin/plant_disease/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/train"
valid_path = "/home/thasin/plant_disease/dataset/new plant diseases dataset(augmented)/New Plant Diseases Dataset(Augmented)/valid"
test_path = "/home/thasin/plant_disease/dataset/test/test"

train_df = generate_image_dataframe(train_path)
valid_df = generate_image_dataframe(valid_path)
test_df = generate_image_dataframe(test_path, is_test=True)



In [ ]:
print(pathlib)

In [6]:
train_df.head()

,img_path,label
0,/home/thasin/plant_disease/dataset/new plant d...,healthy
1,/home/thasin/plant_disease/dataset/new plant d...,mildew
2,/home/thasin/plant_disease/dataset/new plant d...,mildew
3,/home/thasin/plant_disease/dataset/new plant d...,mildew
4,/home/thasin/plant_disease/dataset/new plant d...,mildew


In [8]:
test_df.head()

,img_path,label
0,/home/thasin/plant_disease/dataset/test/test/C...,CornCommonRust1
1,/home/thasin/plant_disease/dataset/test/test/T...,TomatoYellowCurlVirus5
2,/home/thasin/plant_disease/dataset/test/test/P...,PotatoEarlyBlight1
3,/home/thasin/plant_disease/dataset/test/test/T...,TomatoEarlyBlight2
4,/home/thasin/plant_disease/dataset/test/test/T...,TomatoHealthy4


In [9]:
train_df.shape

(2447, 2)

In [17]:
convert_to_int = dict(zip(training_data["label"].unique(),range(len(training_data["label"].unique()))))

In [18]:
convert_to_int

{'healthy': 0, 'mildew': 1, 'blight': 2, 'Blight': 3, 'spot': 4, 'scorch': 5}

In [21]:
range(len(training_data['label'].unique()))

range(0, 6)

In [19]:
training_data["label"].replace(to_replace=convert_to_int.keys(),value=convert_to_int.values(),inplace=True)

In [20]:
training_data.head(10)

,img_path,label
0,/home/thasin/plant_disease/dataset/new plant d...,0
1,/home/thasin/plant_disease/dataset/new plant d...,1
2,/home/thasin/plant_disease/dataset/new plant d...,1
3,/home/thasin/plant_disease/dataset/new plant d...,1
4,/home/thasin/plant_disease/dataset/new plant d...,1
5,/home/thasin/plant_disease/dataset/new plant d...,1
6,/home/thasin/plant_disease/dataset/new plant d...,1
7,/home/thasin/plant_disease/dataset/new plant d...,1
8,/home/thasin/plant_disease/dataset/new plant d...,1
9,/home/thasin/plant_disease/dataset/new plant d...,1


In [22]:
validation_data.replace(to_replace=convert_to_int.keys(),value=convert_to_int.values(),
                     inplace=True)

In [23]:
validation_data.head()

,img_path,label
0,/home/thasin/plant_disease/dataset/new plant d...,1
1,/home/thasin/plant_disease/dataset/new plant d...,1
2,/home/thasin/plant_disease/dataset/new plant d...,1
3,/home/thasin/plant_disease/dataset/new plant d...,1
4,/home/thasin/plant_disease/dataset/new plant d...,1


In [24]:
Y_true_train = to_categorical(y=training_data["label"],num_classes=6)
Y_true_test = to_categorical(y=validation_data["label"],num_classes=6)

In [26]:
Y_true_test,Y_true_train

(array([[0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        ...,
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1.]], dtype=float32),
 array([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        ...,
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1.]], dtype=float32))

In [27]:
Y_true_train.shape, Y_true_test.shape

((2447, 6), (599, 6))

In [29]:
def multiclass_dnn():

    input_to_dnn = Input(shape=(6,))
    first_dense_out = Dense(units=16,activation="relu")(input_to_dnn)
    second_dense_out = Dense(units=8,activation="relu")(first_dense_out)
    second_dense_out = BatchNormalization()(second_dense_out)
    output = Dense(units=6,activation="softmax")(second_dense_out)

    return Model(inputs=[input_to_dnn],outputs=[output])

In [30]:
def custom_data_generator(data_df,Y_true,mb_size):
    
    for time_step in range(data_df.shape[0]//mb_size):
        X_mb = list()
        for img_path in data_df.iloc[time_step*mb_size:(time_step+1)*mb_size,0]:
            img_np_array = plt.imread(img_path)
            reshaped_np_array = img_np_array.reshape(6,)
            X_mb.append(reshaped_np_array)
        X_mb = np.array(X_mb)
        Y_true_mb = Y_true[time_step*mb_size:(time_step+1)*mb_size]
        
        yield X_mb, Y_true_mb

In [31]:
model = multiclass_dnn()
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 6)]               0         
                                                                 
 dense (Dense)               (None, 16)                112       
                                                                 
 dense_1 (Dense)             (None, 8)                 136       
                                                                 
 batch_normalization (Batch  (None, 8)                 32        
 Normalization)                                                  
                                                                 
 dense_2 (Dense)             (None, 6)                 54        
                                                                 
Total params: 334 (1.30 KB)
Trainable params: 318 (1.24 KB)
Non-trainable params: 16 (64.00 Byte)
_____________________________

In [32]:
epochs = 10
training_data_mb_size = 32 
testing_data_mb_size = 32

In [36]:
def loss_fn(Y_true_mb,Y_pred_mb):
    return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y_true=Y_true_mb,y_pred=Y_pred_mb))
optimizer = SGD()

In [37]:
@tf.function
def training_step(X_train_mb,Y_true_train_mb):

    with tf.GradientTape() as tape:
            
        Y_pred_train_mb = model(X_train_mb, training=True)
        training_loss = loss_fn(Y_true_train_mb, Y_pred_train_mb)

    grads = tape.gradient(training_loss, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))

    train_acc_metric.update_state(Y_true_train_mb,Y_pred_train_mb)

    return training_loss

In [38]:
@tf.function
def testing_forward_pass(X_test_mb,Y_true_test_mb):

    Y_pred_test_mb = model(X_test_mb,training=False)
    testing_loss = loss_fn(Y_true_test_mb,Y_pred_test_mb)
    test_acc_metric.update_state(Y_true_test_mb,Y_pred_test_mb)

    return testing_loss

In [39]:
train_acc_metric = tf.keras.metrics.CategoricalAccuracy()
test_acc_metric = tf.keras.metrics.CategoricalAccuracy()

for epoch in range(epochs):

    training_data_generator = custom_data_generator(training_data,Y_true_train,32)

    for time_step, (X_train_mb, Y_true_train_mb) in enumerate(training_data_generator):
        training_loss = training_step(X_train_mb,Y_true_train_mb)

        if (time_step+1) % 50 == 0:
            print("Epoch %d, Time Step %d, Training loss for one mini batch: %.4f"
            % (epoch+1, time_step+1, float(training_loss)))
            
    training_acc = train_acc_metric.result()    
    print("Epoch %d, Training Accuracy: %.2f" % (epoch+1,float(training_acc)))
    train_acc_metric.reset_states()

    testing_data_generator = custom_data_generator(testing_data,Y_true_test,testing_data_mb_size)

    for X_test_mb, Y_true_test_mb in testing_data_generator:
        testing_loss = testing_forward_pass(X_test_mb,Y_true_test_mb)

    print("\nEpoch %d, Testing Loss for last mini batch: %.4f" % (epoch+1,float(testing_loss)))
    testing_acc = test_acc_metric.result()
    print("Epoch %d, Testing Accuracy: %.2f" % (epoch+1,float(testing_acc)))
    test_acc_metric.reset_states()

    print("\n\n")


ValueError: cannot reshape array of size 196608 into shape (6,)